# Rebuild the groove

**Groove id:** rosanna  
**Category:** fidelity  
**Group:** [removed]

Use `drumkit.py` and the Python standard library. Keep `play(...)` as the **last cell**: it is what we listen to.

In [ ]:
from IPython.display import Audio
from drumkit import *

def play(wave, sample_rate=SR):
    return Audio(wave, rate=sample_rate, normalize=False, autoplay=False)

## Tempo and grid

In [ ]:
bpm = 85
step_s = 60 / bpm / 12 # 12 steps per bar
step_samples = round(step_s * SR)

## Patterns

One track per instrument. Replace this example with what you hear.

In [ ]:
def ghost(sample_rate=SR):
    return snare(amplitude=0.04, noise_gain=0.4, sample_rate=sample_rate)


def short_kick(sample_rate=SR):
    return kick(duration=0.085, sample_rate=sample_rate)


def short_snare(sample_rate=SR):
    return snare(duration=0.05, sample_rate=sample_rate)


def crash(sample_rate=SR):
    return open_hat(duration=0.5, tau=0.15, amplitude=0.10, sample_rate=sample_rate)


def tom(f, dur):
    def hit(sample_rate=SR):
        body = kick(
            duration=dur, start_hz=f * 1.6, end_hz=f,
            pitch_tau=0.02, amp_tau=0.04,
            amplitude=0.18, sample_rate=sample_rate,
        )
        stick = edge_fade(
            vca(
                noise(dur, amplitude=0.10, sample_rate=sample_rate),
                envelope_exp(dur, tau=0.004, sample_rate=sample_rate),
            ),
            sample_rate=sample_rate,
        )
        return mix(body, stick)
    return hit


high_tom = tom(150, 0.055)

hat_pattern =        ("x...x.x...x." + "x...x.x...x." + "x...x.x...x." + "x...x.x...x.")
kick_pattern =       ("x........x.." + "......x....." + "...x........" + "............")
short_kick_pattern = ("............" + "............" + "............" + ".........x..")
snare_pattern =      ("............" + "x..........." + "............" + "x...........")
ghost_pattern =      ("............" + "............" + "............" + "........x...")
crash_pattern =      ("x..........." + "............" + "............" + "............")

tracks = [
    ring_track(hat_pattern, closed_hat, step_samples),
    ring_track(kick_pattern, kick, step_samples),
    ring_track(short_kick_pattern, short_kick, step_samples),
    ring_track(snare_pattern, snare, step_samples),
    ring_track(ghost_pattern, ghost, step_samples),
]

bar = render_timed_tracks(tracks)

bar_crash = render_timed_tracks(tracks + [ring_track(crash_pattern, crash, step_samples)])

pickup = render_timed_tracks([
    ring_track("..x.....", short_snare, step_samples),
    ring_track("....x...", short_snare, step_samples),
    ring_track("......x.", high_tom, step_samples),
])

## Listen

In [ ]:
play(pickup + bar_crash + bar + bar_crash + bar*3)